## Homework 6 — Yellow Taxi November 2025
Prototype notebook for Spark batch processing. Covers data loading, repartitioning, and SQL queries.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("yellow_tripdata_2025-11") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

#### Load & Inspect Raw Data

In [ ]:
# load & inspect raw data
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
input_path = RAW_DIR / "yellow_tripdata_2025-11.parquet"

trips_df = spark.read.parquet(str(input_path))
trips_df.printSchema()
trips_df.show(5, truncate=False)

In [ ]:
total_records = trips_df.count()
print(f"Total records (Nov 2025): {total_records:,}")

#### Q2: Average Partition File Size
Repartition into 4 files, write to the Gold layer, then inspect output sizes via terminal:
```bash
ls -lh data/pq/yellow/2025/11/
```

In [ ]:
# ── Q2: Average Partition File Size ────────────────────────── 
# Repartition into 4 files and inspect output size. 
# $ ls -lh data/pq/yellow/2025/11/ 
# Each part file: ~24MB → Answer: 25MB
PQ_DIR = DATA_DIR / "pq" / "yellow" / "2025" / "11"

trips_df.repartition(4) \
    .write.parquet(str(PQ_DIR), mode="overwrite")

print(f"✓ Written to {PQ_DIR}")

**Result:** Each part file is ~24MB → Answer: **25MB**

In [ ]:
# register temp view for SQL queries
trips_df.createOrReplaceTempView("yellow_tripdata")

print("✓ Temp view 'yellow_tripdata' registered")